<a href="https://colab.research.google.com/github/thanatchasan/Ge234/blob/main/Lab3_Functions_6606614714.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🌍 Lab 3: การเขียนฟังก์ชัน
## วิชา GE 234 Basic Programming for Geographers

### 🎯 **วัตถุประสงค์**
1. เข้าใจแนวคิดเกี่ยวกับ **ฟังก์ชัน (Function)** และข้อดีของการใช้ฟังก์ชัน
2. สามารถเขียนฟังก์ชันที่มีพารามิเตอร์และค่าที่คืนกลับ (Return Value) ได้
3. ใช้ฟังก์ชันในการประมวลผลข้อมูลทางภูมิศาสตร์ เช่น การคำนวณระยะทางระหว่างจุด, การแปลงพิกัด, และการวิเคราะห์ค่า NDVI
4. สามารถใช้ **Lambda Function** และ **Recursive Function** ในบริบทของภูมิศาสตร์ได้

---

## 🔹 แบบฝึกหัด 1: สร้างฟังก์ชัน convert_to_utm(lat, lon) เพื่อแปลงค่าพิกัดจาก Lat/Lon เป็น UTM


In [19]:
# import
from pyproj import Transformer
import pandas as pd

# ฟังก์ชันแปลง Lat/Lon -> UTM
def convert_to_utm(lat, lon):
    zone = int((lon + 180) / 6) + 1

    # เลือก EPSG ตาม hemisphere
    epsg = 32600 + zone if lat >= 0 else 32700 + zone

    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    easting, northing = transformer.transform(lon, lat)

    return easting, northing, zone


# ===== ค่าพิกัด(สุพรรณบุรี) =====
lat = 14.4742
lon = 100.1177

easting, northing, zone = convert_to_utm(lat, lon)

print("UTM Easting:", easting)
print("UTM Northing:", northing)
print("Zone:", zone)


# ===== ใช้งานกับ DataFrame (หลายจุดในสุพรรณบุรี) =====
data = {
    "lat": [14.4742, 14.4700, 14.5000],
    "lon": [100.1177, 100.1000, 100.1500]
}

df = pd.DataFrame(data)

# แปลงทั้งคอลัมน์
df[["UTM_E", "UTM_N", "Zone"]] = df.apply(
    lambda row: pd.Series(convert_to_utm(row["lat"], row["lon"])),
    axis=1
)

print(df)

UTM Easting: 620456.4137427734
UTM Northing: 1600465.1943742677
Zone: 47
       lat       lon          UTM_E         UTM_N  Zone
0  14.4742  100.1177  620456.413743  1.600465e+06  47.0
1  14.4700  100.1000  618550.876092  1.599991e+06  47.0
2  14.5000  100.1500  623923.513305  1.603336e+06  47.0



## 🔹 แบบฝึกหัด 2: เขียนฟังก์ชัน filter_locations_by_lat(locations, min_lat) เพื่อตรวจสอบว่าจังหวัดไหนอยู่เหนือค่าละติจูดที่กำหนด


In [ ]:
def filter_locations_by_lat(locations, min_lat):
    """
    ตรวจสอบว่าจังหวัดไหนอยู่เหนือค่าละติจูดที่กำหนด
    อาร์กิวเมนต์:
        locations (list of dict): รายการของพจนานุกรม แต่ละพจนานุกรมมีคีย์ 'name' และ 'lat'
        min_lat (float): ค่าละติจูดขั้นต่ำ
    คืนค่า:
        list of dict: รายการของสถานที่ที่อยู่เหนือค่าละติจูดที่กำหนด
    """
    filtered_locations = []
    for location in locations:
        if location['lat'] > min_lat:
            filtered_locations.append(location)
    return filtered_locations

# ตัวอย่างจังหวัดในประเทศไทย
thai_provinces = [
    {'name': 'กรุงเทพมหานคร', 'lat': 13.7563, 'lon': 100.5018},
    {'name': 'เชียงใหม่', 'lat': 18.7883, 'lon': 98.9853},
    {'name': 'ขอนแก่น', 'lat': 16.4444, 'lon': 102.8361},
    {'name': 'ภูเก็ต', 'lat': 7.8804, 'lon': 98.3923},
    {'name': 'สงขลา', 'lat': 7.2025, 'lon': 100.5960},
    {'name': 'ระยอง', 'lat': 12.6841, 'lon': 101.2131},
    {'name': 'นครราชสีมา', 'lat': 14.9754, 'lon': 102.1004},
    {'name': 'ปทุมธานี', 'lat': 14.02, 'lon': 100.52}
]

# กำหนดค่าละติจูดขั้นต่ำ
min_latitude = 10.0 #

# เรียกใช้ฟังก์ชันเพื่อกรองจังหวัดที่อยู่เหนือค่าละติจูดที่กำหนด
northern_provinces = filter_locations_by_lat(thai_provinces, min_latitude)

print(f"จังหวัดที่อยู่เหนือละติจูด {min_latitude} องศาเหนือ:")
if northern_provinces:
    for province in northern_provinces:
        print(f"- {province['name']} (ละติจูด: {province['lat']})")
else:
    print("ไม่พบจังหวัดใดๆ ที่อยู่เหนือละติจูดที่กำหนด")

จังหวัดที่อยู่เหนือละติจูด 10.0 องศาเหนือ:
- กรุงเทพมหานคร (ละติจูด: 13.7563)
- เชียงใหม่ (ละติจูด: 18.7883)
- ขอนแก่น (ละติจูด: 16.4444)
- ระยอง (ละติจูด: 12.6841)
- นครราชสีมา (ละติจูด: 14.9754)
- ปทุมธานี (ละติจูด: 14.02)



## 🔹 แบบฝึกหัด 3: ใช้ Lambda Function คำนวณระยะทางระหว่างสองจุด


In [ ]:
import math

distance_lambda = lambda lat1, lon1, lat2, lon2: haversine(lat1, lon1, lat2, lon2)

# ทดสอบฟังก์ชัน Lambda สำหรับคำนวณระยะทาง
lat1_test, lon1_test = 14.02, 100.52   # ปทุมธานี
lat2_test, lon2_test = 7.2025, 100.5960 # สงขลา

distance_from_lambda = distance_lambda(lat1_test, lon1_test, lat2_test, lon2_test)
print(f"ระยะทางระหว่างปทุมธานีกับสงขลา (Lambda): {distance_from_lambda:.2f} กิโลเมตร")

ระยะทางระหว่างปทุมธานีกับสงขลา (Lambda): 758.12 กิโลเมตร



## 🔹 แบบฝึกหัด 4: ใช้ Recursive Function คำนวณลำดับฟีโบนัชชี (Fibonacci)

In [ ]:
def fibonacci(n):
    """ คำนวณลำดับฟีโบนัชชีโดยใช้ recursive function """
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

# ทดสอบฟังก์ชัน
print(f"ลำดับ Fibonacci ตัวที่ 7 คือ {fibonacci(7)}")
print(f"ลำดับ Fibonacci ตัวที่ 0 คือ {fibonacci(0)}")
print(f"ลำดับ Fibonacci ตัวที่ 1 คือ {fibonacci(1)}")
print(f"ลำดับ Fibonacci ตัวที่ 10 คือ {fibonacci(10)}")

ลำดับ Fibonacci ตัวที่ 7 คือ 13
ลำดับ Fibonacci ตัวที่ 0 คือ 0
ลำดับ Fibonacci ตัวที่ 1 คือ 1
ลำดับ Fibonacci ตัวที่ 10 คือ 55



6606614714 ธนัชชา สันติเตชกุล